### Visualize elements of the data set by performing verlet integration on the accelerometer data.

#### Motivation

In generative AI, it can be difficult to determine the quality of a model. In the case of image generation models, it is easy to see by simply looking at the generated images and intuitively judging how well it matches the data it is supposed to model. However, with the given data this is not easily possible, therefore I have created a script to visualize data points.

#### Mathematical Foundation

There are two problems to solve:
1. Convert acceleration vectors to position vectors.
2. Convert rotational velocity vectors to absolute rotation vectors.

##### Acceleration to Position

Verlet integration lends itself well to this. It conserves energy better than standard Euler integration over the long term. As the time series are very short, it probably doesn't make a big difference in this task, but its implementation is also very straightforward. Let $x(t)$ be the position at time t, and assume it is twice differentiable, then performing a Taylor expansion forwards and backwards, we get:

$x(t + \triangle t) = x(t) + \dot x(t) \triangle t + \frac{1}{2} \ddot x(t) \triangle t^2$ 

and also 

$x(t - \triangle t) = x(t) - \dot x(t) \triangle t + \frac{1}{2} \ddot x(t) \triangle t^2$

Then $x(t + \triangle t) + x(t - \triangle t) = 2x(t) + \ddot x(t) \triangle t^2 \iff x(t + \triangle t) = \ddot x(t) \triangle t^2 - x(t - \triangle t) + 2x(t)$

Converting this to the discrete case, we get the recurrence $x_{n + 1} = a_n \triangle t^2 - x_{n - 1} + 2 x_n$, where:
- $x_n$ is the position at time $n$
- $\triangle t$ is the timestep
- $a_n$ is the acceleration at time n, which we have given in the dataset.

Hereby, we define the initial position $x_0 := (0, 0, 0)$. In order to be able to evaluate the recurrence we also lack the $x_1$, i.e. we lack information about the initial velocity. If the braking measurements always stop at velocity 0, we can retrieve the start velocity by performing this process in reverse. But in this case this can not be guaranteed, therefore, I will instead take the possibly inaccurate starting velocity from the metadata (scalar) $v_{\text{start}}$ and set $x_1 = x_0 + (v_{\text{start}}, 0, 0) \triangle t$, setting the coordinate system such that the vehicle initially moves in the positive x direction.

##### Rotational Velocity to Rotation

Here, we have given the first derivative, therefore Verlet integration isn't effective, therefore we will use naive Euler integration. Let $r(t)$ be the rotation at time t. We have 

$r(t + \triangle t) = r(t) + \dot r(t) \triangle t$

Or in the discrete case:

$r_{n+1} = r_n + v_{n} * \triangle t$

$r_0$ is unknown. The car will be assumed to be rotated towards its initial travelling velocity (towards positive x).

This integrators are implemented in code as follows:

In [1]:
import numpy as np

def positional_verlet_integrate(a_n, dt, x_0=np.array([0, 0, 0]), v_start=np.array([0, 0, 0])):
    positions = [x_0, x_0 + v_start * dt]

    for acceleration in a_n:
        positions.append(acceleration * dt **2 - positions[-2] + 2 * positions[-1])

    return np.array(positions)

def rotational_euler_integrate(v_n, dt, r_0=np.array([1, 0, 0])):
    rotations = [r_0]

    for velocity in v_n:
        rotations.append(rotations[-1] + velocity * dt)

    return rotations

def numeric_derivative(p_n, dt):
    v_n = []
    
    for prev, curr in zip(p_n, p_n[1:]):
        v_n.append((prev - curr) / dt)

    return np.array(v_n)


#### Visualization
For the visualization, we create an animation using the Manim Python package by Grant Sanderson (3Blue1Brown). The code for this is in the visualize.py module for reusability.

Let's load an example from the dataset, integrate it and visualize it.

In [ ]:
import importlib
import visualize
import pickle
from IPython.display import Video
importlib.reload(visualize)

with open('../../data/train.pickle', 'rb') as f:
    train = pickle.load(f)

# --- Extract & Process Data ---
example = train.iloc[111]
print(example)
visualize.visualize(example, dt=1/80, scale=1.7)
Video("media/videos/480p15/car_trajectory.mp4", embed=True)


C:\Users\linus\AppData\Local\Temp\ipykernel_14244\542150319.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  train = pickle.load(f)


sensor_data             [[-5.79520190612277, 0.07869257228584588, -0.0...
label                                                                   3
model                                                             Golf VI
velocity                                                               20
mass                                                               1341.0
deceleration_average                                             -5.07987
Name: 111, dtype: object


[06/19/25 04:47:20] INFO     Animation 0 : Partial movie file written in                   ]8;id=442456;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=801026;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py#588\588]8;;\
                             'C:\Users\linus\Documents\GitHub\ss2025-srw-nn\visualizer\med                         
                             ia\videos\480p15\partial_movie_files\CarTrajectory\1185818338                         
                             _466977772_30496077.mp4'                                                              

                    INFO     Animation 1 : Partial movie file written in                   ]8;id=256419;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=869056;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py#588\588]8;;\
                             'C:\Users\linus\Documents\GitHub\ss2025-srw-nn\visualizer\med                         
                             ia\videos\480p15\partial_movie_files\CarTrajectory\624642324_                         
                             1754310298_830070352.mp4'                                                             

                    INFO     Combining to Movie file.                                      ]8;id=35420;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=895183;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py#739\739]8;;\

                    INFO                                                                   ]8;id=62371;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=250917;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py#886\886]8;;\
                             File ready at                                                                         
                             'C:\Users\linus\Documents\GitHub\ss2025-srw-nn\visualizer\med                         
                             ia\videos\480p15\car_trajectory.mp4'                                                  
                                                                                                                   

                    INFO     Cache flushed. 3 file(s) deleted in                           ]8;id=199527;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=321432;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene_file_writer.py#870\870]8;;\
                             C:\Users\linus\Documents\GitHub\ss2025-srw-nn\visualizer\medi                         
                             a\videos\480p15\partial_movie_files\CarTrajectory.                                    

                    INFO     Rendered CarTrajectory                                                    ]8;id=985892;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene.py\scene.py]8;;\:]8;id=865832;file://c:\Users\linus\AppData\Local\Programs\Python\Python313\Lib\site-packages\manim\scene\scene.py#255\255]8;;\
                             Played 2 animations                                                                   